# Flight Maneuver Classification - Training Pipeline

Main training workflow with proper train/validation split, feature selection, hyperparameter tuning, and model comparison.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')
from utils import extract_features
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score,
    precision_recall_fscore_support, accuracy_score
)
import xgboost as xgb
import lightgbm as lgb
import warnings
import pickle
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load & Prepare Data

In [ ]:
# Load training data
train_df = pd.read_csv('../data/train_set.csv')
print(f"Train shape: {train_df.shape}")
print(f"Unique maneuvers: {train_df['maneuver_Id'].nunique()}")
print(f"Label distribution:\n{train_df['label'].value_counts().sort_index()}")

## 3. Feature Extraction Function

## 4. Train/Validation Split & Feature Extraction

In [ ]:
# Get unique maneuvers and split 70/30 on maneuver level
maneuvers = train_df['maneuver_Id'].unique()
train_maneuvers, val_maneuvers = train_test_split(
    maneuvers, test_size=0.3, random_state=42,
    stratify=train_df.drop_duplicates('maneuver_Id').set_index('maneuver_Id').loc[maneuvers, 'label'].values
)

print(f"Train maneuvers: {len(train_maneuvers)}")
print(f"Validation maneuvers: {len(val_maneuvers)}")

# Create train and validation dataframes
train_set = train_df[train_df['maneuver_Id'].isin(train_maneuvers)]
val_set = train_df[train_df['maneuver_Id'].isin(val_maneuvers)]

print(f"Train set: {train_set.shape}")
print(f"Validation set: {val_set.shape}")
print(f"\nTrain labels: {train_set['label'].value_counts().sort_index().to_dict()}")
print(f"Validation labels: {val_set['label'].value_counts().sort_index().to_dict()}")

In [ ]:
# Extract features from training set
print("Extracting training features...")
X_train = train_set.groupby('maneuver_Id').apply(extract_features).reset_index()
y_train = train_set.drop_duplicates('maneuver_Id')[['maneuver_Id', 'label']].set_index('maneuver_Id').loc[X_train['maneuver_Id'], 'label'].values
X_train = X_train.fillna(0)
X_train.columns = X_train.columns.astype(str)
X_train_features = X_train.iloc[:, 1:]

print(f"\nTraining features: {X_train_features.shape}")
print(f"Feature types: {X_train_features.dtypes.unique()}")
print(f"Sample features:\n{X_train_features.columns.tolist()[:10]}...")

In [ ]:
# Extract features from validation set
print("Extracting validation features...")
X_val = val_set.groupby('maneuver_Id').apply(extract_features).reset_index()
y_val = val_set.drop_duplicates('maneuver_Id')[['maneuver_Id', 'label']].set_index('maneuver_Id').loc[X_val['maneuver_Id'], 'label'].values
X_val = X_val.fillna(0)
X_val.columns = X_val.columns.astype(str)
X_val_features = X_val.iloc[:, 1:]

print(f"Validation features: {X_val_features.shape}")

## 5. Feature Selection

In [ ]:
# Scale features for feature selection
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_val_scaled = scaler.transform(X_val_features)

print(f"Scaled training shape: {X_train_scaled.shape}")

In [ ]:
# Feature selection using SelectKBest
k_features = 30  # Select top 30 features
feature_selector = SelectKBest(f_classif, k=k_features)
X_train_selected = feature_selector.fit_transform(X_train_scaled, y_train)
X_val_selected = feature_selector.transform(X_val_scaled)

selected_indices = feature_selector.get_support(indices=True)
selected_features = [X_train_features.columns[i] for i in selected_indices]

print(f"\nSelected {len(selected_features)} features out of {X_train_features.shape[1]}")
print(f"\nTop 15 selected features:")
feature_scores = pd.DataFrame({
    'feature': X_train_features.columns,
    'score': feature_selector.scores_
}).sort_values('score', ascending=False)
print(feature_scores.head(15).to_string())

In [ ]:
# Visualize feature scores
plt.figure(figsize=(12, 6))
top_features = feature_scores.head(20)
plt.barh(range(len(top_features)), top_features['score'].values)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Score')
plt.title('Top 20 Features (SelectKBest with f_classif)')
plt.tight_layout()
plt.show()

## 6. Define Models with Pipelines

In [ ]:
# Create pipelines for each model
pipelines = {
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selector', SelectKBest(f_classif, k=30)),
        ('model', xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1,
            verbosity=0
        ))
    ]),
    'LightGBM': Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selector', SelectKBest(f_classif, k=30)),
        ('model', lgb.LGBMClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ]),
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('feature_selector', SelectKBest(f_classif, k=30)),
        ('model', RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

print(f"Created {len(pipelines)} pipeline models")

## 7. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define hyperparameter grids for each model
param_grids = {
    'XGBoost': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 6, 7],
        'model__learning_rate': [0.05, 0.1, 0.15],
        'feature_selector__k': [25, 30, 35]
    },
    'LightGBM': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 6, 7],
        'model__learning_rate': [0.05, 0.1, 0.15],
        'feature_selector__k': [25, 30, 35]
    },
    'RandomForest': {
        'model__n_estimators': [100, 150],
        'model__max_depth': [12, 15, 18],
        'feature_selector__k': [25, 30, 35]
    }
}

print("Hyperparameter grids defined")

In [ ]:
# Grid search for each model
tuned_models = {}
best_params = {}
best_scores = {}

for model_name in pipelines.keys():
    print(f"\n{'='*60}")
    print(f"GridSearchCV for {model_name}...")
    print(f"{'='*60}")
    
    grid = GridSearchCV(
        pipelines[model_name],
        param_grids[model_name],
        cv=3,
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1
    )
    
    grid.fit(X_train_features, y_train)
    
    tuned_models[model_name] = grid.best_estimator_
    best_params[model_name] = grid.best_params_
    best_scores[model_name] = grid.best_score_
    
    print(f"\nBest score: {grid.best_score_:.4f}")
    print(f"Best parameters: {grid.best_params_}")

## 8. Evaluate on Validation Set

In [ ]:
# Evaluate each model on validation set (unseen during training)
val_results = {}

for model_name, model in tuned_models.items():
    print(f"\n{'='*60}")
    print(f"Evaluating {model_name} on validation set")
    print(f"{'='*60}")
    
    y_pred = model.predict(X_val_features)
    precision, recall, f1, support = precision_recall_fscore_support(y_val, y_pred, average=None)
    
    val_results[model_name] = {
        'predictions': y_pred,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'min_f1': min(f1),
        'accuracy': accuracy_score(y_val, y_pred)
    }
    
    print(f"\nAccuracy: {accuracy_score(y_val, y_pred):.4f}")
    print(f"Per-Class F1:")
    for i, f1_score_val in enumerate(f1):
        print(f"  Class {i}: {f1_score_val:.4f}")
    print(f"Minimum F1: {min(f1):.4f}")

## 9. Model Comparison & Selection

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Model': list(val_results.keys()),
    'Training F1': [best_scores[m] for m in val_results.keys()],
    'Validation Accuracy': [val_results[m]['accuracy'] for m in val_results.keys()],
    'Validation Min F1': [val_results[m]['min_f1'] for m in val_results.keys()]
})

comparison = comparison.sort_values('Validation Min F1', ascending=False)
print("\nModel Comparison:")
print(comparison.to_string(index=False))

best_model_name = comparison.iloc[0]['Model']
print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*60}")

In [ ]:
# Confusion matrix for best model
best_model = tuned_models[best_model_name]
y_pred_best = val_results[best_model_name]['predictions']
cm = confusion_matrix(y_val, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Class 0', 'Class 1', 'Class 2'],
            yticklabels=['Class 0', 'Class 1', 'Class 2'])
plt.title(f'Confusion Matrix - {best_model_name} (Validation Set)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 10. Retrain Best Model on Full Training Set

In [ ]:
# Combine train and validation data
X_full = pd.concat([X_train_features, X_val_features], ignore_index=True)
y_full = np.concatenate([y_train, y_val])

print(f"Full training set: {X_full.shape}")
print(f"Label distribution: {pd.Series(y_full).value_counts().sort_index().to_dict()}")

# Retrain best model on full data with best hyperparameters
print(f"\nRetraining {best_model_name} on full training set...")

# Extract best parameters
best_param_dict = best_params[best_model_name]
model_params = {k.replace('model__', ''): v for k, v in best_param_dict.items() if 'model__' in k}
fs_params = {k.replace('feature_selector__', ''): v for k, v in best_param_dict.items() if 'feature_selector__' in k}

# Build model with best params
if best_model_name == 'XGBoost':
    base_model = xgb.XGBClassifier(random_state=42, n_jobs=-1, verbosity=0, **model_params)
elif best_model_name == 'LightGBM':
    base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, **model_params)
else:
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1, **model_params)

# Create final pipeline with best hyperparameters
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selector', SelectKBest(f_classif, **fs_params)),
    ('model', base_model)
])

# Train on full dataset
final_pipeline.fit(X_full, y_full)
print(f"✓ Final model trained on full dataset ({X_full.shape[0]} maneuvers)")

## 11. Save Best Model

In [ ]:
# Save the final pipeline and metadata
import pickle

model_info = {
    'model': final_pipeline,
    'model_name': best_model_name,
    'feature_columns': X_train_features.columns.tolist(),
    'best_params': best_params[best_model_name],
    'validation_results': {
        'accuracy': val_results[best_model_name]['accuracy'],
        'min_f1': val_results[best_model_name]['min_f1'],
        'f1_scores': val_results[best_model_name]['f1'].tolist()
    }
}

with open('../output/best_model.pkl', 'wb') as f:
    pickle.dump(model_info, f)

print(f"✓ Best model saved to output/best_model.pkl")
print(f"\nModel Summary:")
print(f"  Name: {best_model_name}")
print(f"  Validation Accuracy: {val_results[best_model_name]['accuracy']:.4f}")
print(f"  Validation Min F1: {val_results[best_model_name]['min_f1']:.4f}")

## 12. Summary & Export Results

In [ ]:
# Export detailed results
results_summary = {
    'best_model': best_model_name,
    'validation_metrics': val_results[best_model_name],
    'model_comparison': comparison.to_dict('records'),
    'best_hyperparameters': best_params[best_model_name]
}

import json
with open('../output/training_results.json', 'w') as f:
    json.dump({k: v for k, v in results_summary.items() if k != 'validation_metrics'}, f, indent=2)

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Best Model: {best_model_name}")
print(f"Validation Accuracy: {val_results[best_model_name]['accuracy']:.4f}")
print(f"Validation Min F1: {val_results[best_model_name]['min_f1']:.4f}")
print(f"\nNext: Run flight_maneuver_submission.ipynb to generate test predictions")